# 📦 00 — Data Pre-Processing for Retail Intelligence Training

**Run this notebook FIRST, before any training notebook.**

This notebook converts your raw competition files into the organised datasets that
`train_yolov11m.ipynb` and `train_mobilenet_staff.ipynb` expect.

---
## What each source file is used for

| Source File | Used In | Purpose |
|-------------|---------|--------|
| `Stores/Store 1/CAM 1 - zone.mp4` | `train_yolov11m.ipynb` | Frame extraction → person detection training data (zone cam angle) |
| `Stores/Store 1/CAM 2 - zone.mp4` | `train_yolov11m.ipynb` | Frame extraction → person detection training data (second zone angle) |
| `Stores/Store 1/CAM 3 - entry.mp4` | `train_yolov11m.ipynb` + `train_mobilenet_staff.ipynb` | Best view for staff uniform crops (entry camera sees full body at door) |
| `Stores/Store 1/CAM 5 - billing.mp4` | `train_mobilenet_staff.ipynb` | Close-up crops at billing counter — good for staff vs customer at POS |
| `Stores/Store 2/entry 1.mp4` | `train_yolov11m.ipynb` | Frame extraction — different store layout angle |
| `Stores/Store 2/entry 2.mp4` | `train_yolov11m.ipynb` | Frame extraction — wide-angle entry |
| `Stores/Store 2/zone.mp4` | `train_yolov11m.ipynb` | Frame extraction — overhead zone coverage |
| `Stores/Store 2/billing_area.mp4` | `train_mobilenet_staff.ipynb` | Staff at billing counter crops |
| `Stores/*/layout.png` | NOT used in training | Zone polygon reference only (used in `store_layout/zones.json`) |
| `POS_transactions.csv` | NOT used in training | Used only by API `/funnel` endpoint for POS correlation |
| `sample_events.jsonl` | NOT used in training | JSON schema reference only — used by API ingestion tests |

---
## Output Structure After This Notebook

```
/content/drive/MyDrive/purplle/
├── frames/                        ← Raw extracted frames from all clips
│   ├── store1_cam1/               ← 1 frame per 15 frames (~1/sec at 15fps)
│   ├── store1_cam2/
│   ├── store1_cam3/               ← Entry cam — used by both training notebooks
│   ├── store1_cam5/
│   ├── store2_entry1/
│   ├── store2_entry2/
│   ├── store2_zone/
│   └── store2_billing/
├── yolo_dataset/                  ← YOLO training format (auto-labelled)
│   ├── images/
│   │   ├── train/
│   │   └── val/
│   ├── labels/
│   │   ├── train/
│   │   └── val/
│   └── dataset.yaml
└── staff_crops/                   ← Classifier training crops (sorted)
    ├── staff/                     ← Purple uniform → auto-HSV detected
    └── customer/                  ← All other crops
```

## Step 1 — Install Dependencies & Mount Drive

In [ ]:
# Install lightweight dependencies only (no heavy ML frameworks needed here)
!pip install -q ultralytics opencv-python-headless tqdm

from google.colab import drive
drive.mount('/content/drive')

import os, cv2, json, shutil, random
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

print('✅ Dependencies ready')

## Step 2 — Configure Paths

**Upload your competition files to Google Drive first:**
1. Create a folder `MyDrive/purplle/source/` in Google Drive
2. Upload the entire `Stores/` folder into it (preserving the Store 1 / Store 2 subdirectory structure)
3. Upload `POS_transactions.csv` into `MyDrive/purplle/source/`
4. Upload `sample_events.jsonl` into `MyDrive/purplle/source/`

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive/purplle')
SOURCE_ROOT  = DRIVE_ROOT / 'source'          # Where you uploaded the competition files
OUTPUT_ROOT  = DRIVE_ROOT                     # Outputs go here (frames/, yolo_dataset/, staff_crops/)

FRAME_EVERY_N = 15    # Extract 1 frame every N frames (15 = ~1/sec at 15fps clip)
MAX_FRAMES_PER_CLIP = 500  # Cap per clip to keep dataset manageable
# ─────────────────────────────────────────────────────────────────────────────

# Verify source files exist
clips = {
    'store1_cam1':   SOURCE_ROOT / 'Stores/Store 1/CAM 1 - zone.mp4',
    'store1_cam2':   SOURCE_ROOT / 'Stores/Store 1/CAM 2 - zone.mp4',
    'store1_cam3':   SOURCE_ROOT / 'Stores/Store 1/CAM 3 - entry.mp4',
    'store1_cam5':   SOURCE_ROOT / 'Stores/Store 1/CAM 5 - billing.mp4',
    'store2_entry1': SOURCE_ROOT / 'Stores/Store 2/entry 1.mp4',
    'store2_entry2': SOURCE_ROOT / 'Stores/Store 2/entry 2.mp4',
    'store2_zone':   SOURCE_ROOT / 'Stores/Store 2/zone.mp4',
    'store2_billing':SOURCE_ROOT / 'Stores/Store 2/billing_area.mp4',
}

print('Checking source clip files:')
all_ok = True
for name, path in clips.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1e6 if exists else 0
    status = f'✅ {size_mb:.0f} MB' if exists else '❌ NOT FOUND'
    print(f'  {name:22s} {status}')
    if not exists:
        all_ok = False

pos_csv = SOURCE_ROOT / 'POS_transactions.csv'
events_jsonl = SOURCE_ROOT / 'sample_events.jsonl'
print(f'  POS_transactions.csv   {"✅" if pos_csv.exists() else "❌ NOT FOUND"}')
print(f'  sample_events.jsonl    {"✅" if events_jsonl.exists() else "❌ NOT FOUND"}')

if all_ok:
    print('\n✅ All clip files found — ready to extract frames')
else:
    print('\n❌ Some files missing. Upload them to Drive first (see instructions above)')

## Step 3 — Inspect Clips (FPS, Resolution, Duration)

In [ ]:
print(f'{'Clip':<22} {'FPS':>5} {'W':>5} {'H':>5} {'Frames':>8} {'Duration':>10}')
print('─' * 65)
for name, path in clips.items():
    if not path.exists():
        print(f'{name:<22}  (missing)')
        continue
    cap = cv2.VideoCapture(str(path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur   = total / fps if fps > 0 else 0
    cap.release()
    print(f'{name:<22} {fps:>5.1f} {w:>5} {h:>5} {total:>8} {dur:>8.1f}s')

## Step 4 — Extract Raw Frames

Samples 1 frame every `FRAME_EVERY_N` frames from each clip.
Saves as numbered JPEGs into per-camera subdirectories.

In [ ]:
frames_dir = OUTPUT_ROOT / 'frames'
frames_dir.mkdir(parents=True, exist_ok=True)

total_extracted = 0

for name, path in clips.items():
    if not path.exists():
        print(f'  ⚠️  Skipping {name} (missing)')
        continue

    out_dir = frames_dir / name
    out_dir.mkdir(exist_ok=True)

    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    extracted = 0

    with tqdm(total=min(total_frames, MAX_FRAMES_PER_CLIP * FRAME_EVERY_N),
               desc=name, unit='frame') as pbar:
        frame_idx = 0
        while extracted < MAX_FRAMES_PER_CLIP:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % FRAME_EVERY_N == 0:
                fname = out_dir / f'{name}_{extracted:04d}.jpg'
                cv2.imwrite(str(fname), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
                extracted += 1
            frame_idx += 1
            pbar.update(1)

    cap.release()
    total_extracted += extracted
    print(f'  ✅ {name}: {extracted} frames saved to {out_dir}')

print(f'\n✅ Total frames extracted: {total_extracted}')

## Step 5 — Auto-Label Frames with Pretrained YOLOv11m (Pseudo-Labels)

Uses the pretrained COCO-trained YOLOv11m to detect `person` (class 0) in all extracted frames.
This produces YOLO-format `.txt` label files we'll use to fine-tune the model on retail footage.

> **Why pseudo-labels work here**: Even a COCO-trained detector detects people reasonably well in retail.
> Fine-tuning on these pseudo-labels teaches the model to handle retail-specific conditions
> (occlusion, groups, shelf backgrounds, unusual lighting).

In [ ]:
from ultralytics import YOLO

# Load pretrained model for pseudo-labelling
model = YOLO('yolo11m.pt')  # Downloads automatically
print('✅ YOLOv11m pretrained loaded')

CONF_THRESHOLD = 0.35   # Lower than default to catch partially-visible people
PERSON_CLASS = 0

all_image_paths = sorted(frames_dir.glob('*/*.jpg'))
print(f'Total frames to label: {len(all_image_paths)}')

labels_written = 0
frames_with_people = 0

for img_path in tqdm(all_image_paths, desc='Auto-labelling'):
    results = model(str(img_path), conf=CONF_THRESHOLD, classes=[PERSON_CLASS], verbose=False)

    label_path = img_path.with_suffix('.txt')
    lines = []

    for r in results:
        for box in r.boxes:
            if int(box.cls[0]) != PERSON_CLASS:
                continue
            # YOLO format: class cx cy w h (all normalised 0-1)
            cx, cy, bw, bh = box.xywhn[0].tolist()
            lines.append(f'{PERSON_CLASS} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    if lines:
        label_path.write_text('\n'.join(lines))
        labels_written += len(lines)
        frames_with_people += 1

print(f'\n✅ Labelled {frames_with_people}/{len(all_image_paths)} frames')
print(f'   Total person annotations: {labels_written}')
print(f'   Avg persons/frame: {labels_written/max(1,frames_with_people):.1f}')

## Step 6 — Build YOLO Dataset (80/20 Train/Val Split)

In [ ]:
yolo_dir = OUTPUT_ROOT / 'yolo_dataset'
for split in ['train', 'val']:
    (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Only include labelled frames (frames with at least 1 person annotation)
labelled_pairs = [
    (img_path, img_path.with_suffix('.txt'))
    for img_path in frames_dir.glob('*/*.jpg')
    if img_path.with_suffix('.txt').exists()
    and img_path.with_suffix('.txt').stat().st_size > 0
]

random.seed(42)
random.shuffle(labelled_pairs)
split_idx = int(len(labelled_pairs) * 0.8)
train_pairs = labelled_pairs[:split_idx]
val_pairs   = labelled_pairs[split_idx:]

def copy_to_split(pairs, split_name):
    for img_path, lbl_path in tqdm(pairs, desc=f'Copying to {split_name}'):
        shutil.copy(img_path, yolo_dir / 'images' / split_name / img_path.name)
        shutil.copy(lbl_path, yolo_dir / 'labels' / split_name / lbl_path.name)

copy_to_split(train_pairs, 'train')
copy_to_split(val_pairs,   'val')

# Write dataset.yaml
yaml_content = f"""# Retail Intelligence — Person Detection Dataset
# Auto-generated by 00_preprocess_data.ipynb
path: {yolo_dir}
train: images/train
val:   images/val

nc: 1
names:
  0: person
"""
(yolo_dir / 'dataset.yaml').write_text(yaml_content)

print(f'\n✅ YOLO dataset built:')
print(f'   Train images: {len(train_pairs)}')
print(f'   Val   images: {len(val_pairs)}')
print(f'   dataset.yaml: {yolo_dir / "dataset.yaml"}')

## Step 7 — Extract Staff Crops for Classifier Training

Runs YOLO detection on the **entry and billing camera frames** (best views for full-body crops),
then uses HSV colour analysis to auto-sort crops into `staff/` (purple uniform) vs `customer/`.

### Which clips are used for crops:
| Clip | Reason |
|------|--------|
| `store1_cam3` (entry) | Best full-body view — staff visible at doorway |
| `store1_cam5` (billing) | Close-up at POS counter — staff in clear view |
| `store2_billing` (billing) | Same rationale for Store 2 |

Zone cameras (CAM1, CAM2, zone.mp4) are **not used for crops** — overhead/angled views
give poor full-body crops for the staff classifier.

In [ ]:
# Only process entry and billing frames for crop quality
CROP_SOURCE_CAMS = ['store1_cam3', 'store1_cam5', 'store2_billing']

staff_dir    = OUTPUT_ROOT / 'staff_crops' / 'staff'
customer_dir = OUTPUT_ROOT / 'staff_crops' / 'customer'
staff_dir.mkdir(parents=True, exist_ok=True)
customer_dir.mkdir(parents=True, exist_ok=True)

# HSV range for Purplle staff purple/violet uniform
# Purple in HSV: H ≈ 130-165, S > 50, V > 50
PURPLE_HSV_LOW  = np.array([130, 50, 50],  dtype=np.uint8)
PURPLE_HSV_HIGH = np.array([165, 255, 255], dtype=np.uint8)
PURPLE_PIXEL_RATIO_THRESHOLD = 0.10  # 10%+ purple pixels → staff

MIN_CROP_SIZE = 40  # Skip crops smaller than 40x40 px

def is_staff_by_hsv(crop_bgr: np.ndarray) -> bool:
    """Return True if crop contains significant purple (staff uniform)."""
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, PURPLE_HSV_LOW, PURPLE_HSV_HIGH)
    ratio = mask.sum() / 255 / (hsv.shape[0] * hsv.shape[1])
    return ratio >= PURPLE_PIXEL_RATIO_THRESHOLD

staff_count, customer_count = 0, 0

for cam_name in CROP_SOURCE_CAMS:
    cam_dir = frames_dir / cam_name
    if not cam_dir.exists():
        print(f'  ⚠️  {cam_name} frames not found — skipping')
        continue

    img_paths = sorted(cam_dir.glob('*.jpg'))
    print(f'  Processing {cam_name}: {len(img_paths)} frames...')

    for img_path in tqdm(img_paths, desc=cam_name, leave=False):
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue
        h, w = frame.shape[:2]

        results = model(frame, conf=0.40, classes=[PERSON_CLASS], verbose=False)
        for r in results:
            for i, box in enumerate(r.boxes):
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                if (x2 - x1) < MIN_CROP_SIZE or (y2 - y1) < MIN_CROP_SIZE:
                    continue

                crop = frame[y1:y2, x1:x2]
                crop_resized = cv2.resize(crop, (224, 224))

                # Auto-sort by HSV purple detection
                if is_staff_by_hsv(crop):
                    fname = staff_dir / f'{cam_name}_{img_path.stem}_{i:02d}.jpg'
                    cv2.imwrite(str(fname), crop_resized)
                    staff_count += 1
                else:
                    fname = customer_dir / f'{cam_name}_{img_path.stem}_{i:02d}.jpg'
                    cv2.imwrite(str(fname), crop_resized)
                    customer_count += 1

print(f'\n✅ Staff crops extracted:')
print(f'   staff/    : {staff_count} crops')
print(f'   customer/ : {customer_count} crops')
print(f'   Total     : {staff_count + customer_count}')

if staff_count < 30:
    print('\n⚠️  WARNING: Very few staff crops detected automatically.')
    print('   The purple HSV range may need tuning, OR the clips have few staff in frame.')
    print('   Manually move some staff images from customer/ to staff/ before training.')

## Step 8 — Validate POS Transactions CSV

`POS_transactions.csv` is **not used for training** — it's consumed by the API server's
`/funnel` endpoint. This step just validates the file is correct and gives you a preview.

The API server loads it via `POST /admin/pos/reload` when the pipeline container starts.

In [ ]:
import pandas as pd

if not pos_csv.exists():
    print('❌ POS_transactions.csv not found — upload to Drive source folder')
else:
    df = pd.read_csv(pos_csv)
    print(f'✅ POS_transactions.csv loaded: {len(df)} rows')
    print(f'\nSchema: {list(df.columns)}')
    print(f'Stores: {df["store_id"].unique()}')
    print(f'Date format sample: {df["order_date"].iloc[0]}  (expected: DD-MM-YYYY)')
    print(f'Time format sample: {df["order_time"].iloc[0]}  (expected: HH:MM:SS)')
    print(f'Unique orders: {df.groupby(["store_id","order_date","order_time"]).ngroups}')
    print(f'Total line items: {len(df)}')
    print(f'\nBrands present: {sorted(df["brand_name"].dropna().unique())}')

    # Verify DD-MM-YYYY parse works
    test_parse = pd.to_datetime(df['order_date'].iloc[0], format='%d-%m-%Y')
    print(f'\n✅ Date parses correctly: {df["order_date"].iloc[0]} → {test_parse.date()}')
    print('\n⚠️  NOTE: POS data is only for Store ST1008 (Store 2). Store 1 has no POS data.')
    print('   The /funnel endpoint will show 0 purchases for ST1076.')

    display(df.head(5))

## Step 9 — Validate sample_events.jsonl

`sample_events.jsonl` is **not used for training** — it's the ground truth schema reference
for the API ingestion endpoint. This step just verifies you understand the 3 distinct schemas.

In [ ]:
if not events_jsonl.exists():
    print('❌ sample_events.jsonl not found')
else:
    events = [json.loads(line) for line in events_jsonl.read_text().splitlines() if line.strip()]
    print(f'✅ Loaded {len(events)} sample events')
    print()

    # Detect schema type by field fingerprinting
    schema_counts = {'entry_exit': 0, 'zone': 0, 'queue': 0, 'unknown': 0}
    for e in events:
        if 'id_token' in e and 'event_timestamp' in e:
            schema_counts['entry_exit'] += 1
        elif 'track_id' in e and 'zone_id' in e and 'event_time' in e:
            schema_counts['zone'] += 1
        elif 'queue_event_id' in e or 'queue_join_ts' in e:
            schema_counts['queue'] += 1
        else:
            schema_counts['unknown'] += 1

    print('Event schema breakdown:')
    for schema_type, count in schema_counts.items():
        print(f'  {schema_type:<15}: {count} events')

    print()
    print('Schema 1 — Entry/Exit event fields:')
    entry_evt = next(e for e in events if 'event_timestamp' in e)
    for k, v in entry_evt.items():
        print(f'  {k:<20}: {repr(v)}')

    print()
    print('Schema 2 — Zone event fields:')
    zone_evt = next(e for e in events if 'event_time' in e)
    for k, v in zone_evt.items():
        print(f'  {k:<20}: {repr(v)}')

    print()
    print('Schema 3 — Queue event fields:')
    queue_evt = next(e for e in events if 'queue_join_ts' in e)
    for k, v in queue_evt.items():
        print(f'  {k:<20}: {repr(v)}')

## Step 10 — Summary & Next Steps

In [ ]:
print('='*65)
print('PRE-PROCESSING COMPLETE — Summary')
print('='*65)

# Count outputs
train_imgs = list((yolo_dir / 'images' / 'train').glob('*.jpg')) if (yolo_dir / 'images' / 'train').exists() else []
val_imgs   = list((yolo_dir / 'images' / 'val').glob('*.jpg'))   if (yolo_dir / 'images' / 'val').exists()   else []
staff_imgs    = list(staff_dir.glob('*.jpg'))    if staff_dir.exists()    else []
customer_imgs = list(customer_dir.glob('*.jpg')) if customer_dir.exists() else []

print(f'\n📁 YOLO Dataset ({yolo_dir.name}/)')
print(f'   Train images : {len(train_imgs)}')
print(f'   Val   images : {len(val_imgs)}')

print(f'\n📁 Staff Classifier Crops ({(OUTPUT_ROOT / "staff_crops").name}/)')
print(f'   staff/       : {len(staff_imgs)} crops')
print(f'   customer/    : {len(customer_imgs)} crops')

print()
print('─'*65)
print('🚀 NEXT STEPS:')
print()
print('1. ▶ Open train_yolov11m.ipynb')
print('   Input : yolo_dataset/  (just built)')
print('   Output: yolov11m_retail.onnx  → put in project models/')
print()
print('2. ▶ Open train_mobilenet_staff.ipynb')
print('   Input : staff_crops/  (just built)')
print('   Output: mobilenet_staff.onnx  → put in project models/')
print()
print('3. ▶ Download both .onnx files to your local machine')
print('   Place in: d:/desktop/projects/purplle/models/')
print()
print('4. ▶ Restart vision-pipeline container:')
print('   docker compose restart vision-pipeline')
print('='*65)